# Panerai ETL Workflow Test
This notebook implements the incremental ETL pipeline from Bronze $\rightarrow$ Silver $\rightarrow$ Gold layers.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql import Window
import yaml

# --- Configuration ---
ENV = "dev"
BRONZE_TABLE = f"panerai_project.{ENV}.panerai_data_bronze_{ENV}"
SILVER_TABLE = f"panerai_project.{ENV}.panerai_data_silver_{ENV}"
VOLUME_PATH = f"/Volumes/panerai_project/{ENV}/panerai_data"
EXCHANGE_RATES = {
    "USD": 0.88, "JPY": 0.0056, "AED": 0.24, 
    "CHF": 1.06, "GBP": 1.16, "EUR": 1.0
}

# Load YAML for URL corrections
# We load this here so it's available globally in the notebook
try:
    with open("./ref_corrected_url.yaml", "r") as f:
        REF_CORRECTED_URL = yaml.safe_load(f)
    print("✅ Loaded ref_corrected_url.yaml successfully.")
except FileNotFoundError:
    REF_CORRECTED_URL = {}
    print("⚠️ ref_corrected_url.yaml not found. Continuing without corrections.")


## 1. Bronze Layer: Incremental Ingestion
Using Auto Loader to ingest new files from the Volume.

In [ ]:
# In a real production job, we would use spark.readStream with cloudFiles
# For the test notebook, we simulate incremental load by reading the volume
raw_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("recursiveFileLookup", "true")\
    .load(VOLUME_PATH)

# Adding metadata for traceability
# Note: _metadata is a special column available in read_files(), not standard read.csv
# We use input_file_name() to get the source path
bronze_df = raw_df.withColumn("file_path", F.col("_metadata.file_path"))

# --- CRITICAL: Create the Bronze Table ---
# We write the DataFrame to a Delta table so it exists in the catalog
bronze_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(BRONZE_TABLE)

print(f"✅ Bronze table {BRONZE_TABLE} created/updated.")
display(spark.read.table(BRONZE_TABLE).limit(5))

## 2. Silver Layer: 
#### 2.1. Cleaning & Normalization
Processing the raw data and preparing for a MERGE into the Silver table.

In [ ]:
def clean_silver_data(df):
    # 1. Drop rows without reference
    df = df.dropna(subset=["reference"])
    
    # 2. URL Correction using YAML
    if REF_CORRECTED_URL:
        # Flatten YAML structure into a list of tuples for Spark DataFrame
        corrections = [
            (ref, country, data["url"])
            for ref, countries in REF_CORRECTED_URL.items()
            for country, data in countries.items()
        ]
        correct_urls_df = spark.createDataFrame(corrections, ["reference", "country", "correct_url"])
        
        # Join and overwrite URL if a correction exists
        df = df.join(F.broadcast(correct_urls_df), on=["reference", "country"], how="left")
        df = df.withColumn("url", F.coalesce(F.col("correct_url"), F.col("url")))
        df = df.drop("correct_url")
    
    # 3. Clean Numeric Columns
    df = df.withColumn("price_clean", F.regexp_replace(F.col("price"), r"[^0-9]", "").cast("int"))
    df = df.withColumn("diameter_mm", F.regexp_replace(F.col("diameter_tag"), r"[^0-9]", "").cast("int"))
    
    # 4. Currency Conversion
    rate_expr = F.lit(1.0)
    for curr, rate in EXCHANGE_RATES.items():
        rate_expr = F.when(F.col("currency") == curr, F.lit(rate)).otherwise(rate_expr)
    
    df = df.withColumn("price_eur", F.col("price_clean") * rate_expr)
    
    # 5. Missing Value Handling
    df = df.withColumn("edition", F.coalesce(F.col("edition"), F.lit("unknown")))
    df = df.withColumn("comment", F.coalesce(F.col("comment"), F.lit("unknown")))
    
    # 6. Price Display Logic
    df = df.withColumn("price_display", 
                      F.when(F.col("price_clean").isNull(), F.lit("Contact store directly")).otherwise(F.col("price_clean").cast("string")))
    
    # 7. DEDUPLICATION & EXCEPTION HANDLING
    # We now deduplicate after correcting URLs. 
    # If there are still multiple different URLs for the same ref/country/date, 
    # we flag them as exceptions.
    
    # Count unique URLs per reference/country/date
    url_count_window = Window.partitionBy("reference", "country", "date")
    df = df.withColumn("unique_url_count", F.size(F.collect_set("url").over(url_count_window)))
    
    # Flagging logic
    df = df.withColumn("is_exception", F.when(F.col("unique_url_count") > 1, True).otherwise(False))
    
    # Final Deduplication: Keep the first record found
    # We use monotonically_increasing_id to keep it deterministic
    dedup_window = Window.partitionBy("reference", "country", "date").orderBy(F.monotonically_increasing_id().desc())
    df = df.withColumn("row_num", F.row_number().over(dedup_window)) \
           .filter(F.col("row_num") == 1) \
           .drop("row_num")
    
    return df

silver_processed_df = clean_silver_data(bronze_df)
display(silver_processed_df.limit(5))

#### 2.2. Incremental Merge
Instead of Overwriting, we Merge into the target table.

In [ ]:
# First time setup: Create table with the correct schema if it doesn't exist
if not spark.catalog.tableExists(SILVER_TABLE):
    print(f"Initializing {SILVER_TABLE} with current schema...")
    silver_processed_df.write.format("delta").saveAsTable(SILVER_TABLE)
else:
    print(f"Table {SILVER_TABLE} already exists. Proceeding to merge.")

# Fix for Serverless/Modern Runtime: 
# Instead of spark.conf.set, we handle schema evolution at the table level or via the MERGE operation.
# In most modern Databricks runtimes, auto-merge is handled internally or via:
spark.sql(f"ALTER TABLE {SILVER_TABLE} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")

# To perform a MERGE, we need a temporary view
silver_processed_df.createOrReplaceTempView("updates")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING updates AS source
ON target.reference = source.reference 
   AND target.country = source.country 
   AND target.date = source.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("Silver layer merged successfully.")

## 3. Gold Layer: Star Schema
Transforming the flat Silver table into Dimensions and Facts.

In [ ]:
# --- GOLD LAYER PRODUCTION IMPLEMENTATION ---
# In production, we create permanent Delta tables instead of temporary DataFrames

# 1. Dimension: Collection
dim_collection = silver_final.select("collection").distinct() \
    .withColumn("collection_id", F.monotonically_increasing_id())

dim_collection.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"panerai_project.{ENV}.dim_collection")

# 2. Dimension: Country
dim_country = silver_final.select("country", "currency").distinct() \
    .withColumn("country_id", F.monotonically_increasing_id())

dim_country.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"panerai_project.{ENV}.dim_country")

# 3. Dimension: Watch
# We only keep the essential attributes of the watch
dim_watch = silver_final.select("reference", "title", "edition", "comment").distinct()

dim_watch.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"panerai_project.{ENV}.dim_watch")

# 4. Fact Table: Pricing
# Join the silver table with the dimensions to get the IDs
fact_pricing = silver_final.join(dim_collection, "collection") \
                           .join(dim_country, "country") \
                           .select(
                               "reference", 
                               "date", 
                               "price_eur", 
                               "diameter_mm", 
                               "collection_id", 
                               "country_id"
                           )

fact_pricing.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"panerai_project.{ENV}.fact_pricing")

print("✅ Gold Layer: All Dimension and Fact tables have been saved successfully.")
display(spark.read.table(f"panerai_project.{ENV}.fact_pricing").limit(5))